# VoyageAI


In [1]:
%pip install -qU voyageai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
EMBEDDING_MODEL = "voyage-large-2-instruct"
DIMENSIONS = 1024

In [3]:
import voyageai
from dotenv import load_dotenv

load_dotenv()

vo = voyageai.Client()

result = vo.embed(["hello world"], model=EMBEDDING_MODEL, input_type="document")

result.embeddings[0][:4]

[0.03380299732089043,
 0.010667615570127964,
 -0.016215115785598755,
 -0.004741863813251257]

In [4]:
assert len(result.embeddings[0]) == DIMENSIONS

In [5]:
from tqdm import tqdm

from typing import Literal


def get_embeddings(texts: list[str], input_type: str = "document") -> list[list[float]]:
    result = vo.embed(texts, model=EMBEDDING_MODEL, input_type=input_type)
    return result.embeddings


def get_embeddings_batched(
    texts: list[str],
    batch_size: int = 128,
    input_type: Literal["document", "query"] = "document",
    show_progress: bool = True,
) -> list[list[float]]:
    all_embeddings = []

    # Create iterable for tqdm
    batches = range(0, len(texts), batch_size)

    # Wrap with tqdm if show_progress is True
    if show_progress:
        batches = tqdm(batches, total=len(batches), desc="Getting embeddings")

    for i in batches:
        batch = texts[i : i + batch_size]
        batch_embeddings = get_embeddings(batch, input_type=input_type)
        all_embeddings.extend(batch_embeddings)

    return all_embeddings

# MongoDB


In [6]:
from dotenv import load_dotenv
import os
import pymongo


load_dotenv()

mongo_client = pymongo.MongoClient(os.getenv("MONGODB_URI"))

db = mongo_client["blogdb"]

collection = db["ai_news"]

collection.find_one()

{'_id': ObjectId('667d1ef6fc45eb48396a6a0c'),
 'date': datetime.datetime(2024, 6, 20, 14, 0),
 'title': "Anthropic's rivalry with OpenAI heats up with its claim new Claude AI surpasses GPT-4o",
 'body': 'Just a month after OpenAI rolled out its latest AI model, GPT-4o, today its rival Anthropic—famously founded by breakaway OpenAI researchers in 2021—said it had developed a new model to top it.',
 'url': 'https://www.msn.com/en-us/news/technology/anthropic-s-rivalry-with-openai-heats-up-with-its-claim-new-claude-ai-surpasses-gpt-4o/ar-BB1oApe1',
 'image': 'https://img-s-msn-com.akamaized.net/tenant/amp/entityid/BB1oAbpV.img?w=2048&h=1366&m=4&q=88',
 'source': 'Fortune on MSN.com',
 'found_at': datetime.datetime(2024, 6, 26, 9, 51, 56, 553000),
 'region': 'wt-wt',
 'domain': 'msn.com'}

# Pinecone


In [7]:
%pip install -q pinecone-client[grpc]

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
protoc-gen-openapiv2 0.0.1 requires protobuf>=4.21.0, but you have protobuf 3.20.3 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 24.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
from dotenv import load_dotenv

load_dotenv()

pc = Pinecone()

In [11]:
INDEX_NAME = "ai-news-index"

if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSIONS,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws", region="us-east-1"
        ),  # eu-west-1 not available in free plan
    )

index = pc.Index(INDEX_NAME)

# Upsert all data from MongoDB


In [13]:
from pinecone.core.grpc.protos.vector_service_pb2 import UpsertResponse
from pinecone.grpc import GRPCIndex


def upsert_articles_batched(
    index: GRPCIndex,
    ids: list[str],
    embeddings: list[list[float]],
    metadatas: list[dict],
    batch_size: int = 128,
    show_progress: bool = True,
) -> UpsertResponse:
    assert all(len(vector) == DIMENSIONS for vector in embeddings)
    assert len(ids) == len(embeddings) == len(metadatas)

    # create list of (id, embedding, metadata) tuples to be upserted
    data = list(zip(ids, embeddings, metadatas))
    return index.upsert(
        data,
        batch_size=batch_size,
        show_progress=show_progress,
        async_req=False,
    )  # type: ignore

In [37]:
# Function to process and upsert articles
from pinecone.core.grpc.protos.vector_service_pb2 import UpsertResponse


def process_and_upsert_articles() -> UpsertResponse:
    # Fetch all articles from MongoDB
    articles = list(collection.find())

    ids = [str(article["_id"]) for article in articles]

    texts = [f"{article['title']}\n\n{article['body']}" for article in articles]
    embeddings = get_embeddings_batched(texts=texts, input_type="document")

    metadatas = [
        {
            "title": article["title"],
            "url": article["url"],
            "body": article["body"],
            "found_at": article["found_at"].timestamp(),
            "date": article["date"].timestamp(),
            # TODO : add region !
        }
        for article in articles
    ]

    return upsert_articles_batched(
        index,
        ids,
        embeddings,
        metadatas,
        show_progress=True,
    )


# TODO : only upsert articles not in pinecone
# # Run the process
# process_and_upsert_articles()

Getting embeddings: 100%|██████████| 281/281 [05:20<00:00,  1.14s/it]


Upserted vectors:   0%|          | 0/35941 [00:00<?, ?it/s]

upserted_count: 35941

In [14]:
print(index.describe_index_stats())

{'dimension': 1024,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 35941}},
 'total_vector_count': 35941}


# Test queries


In [17]:
from datetime import datetime, timedelta

QUERY = "tesla FSD V12"

query_embedding = get_embeddings([QUERY], input_type="query")[0]

published_date_start = int((datetime.now() - timedelta(days=10)).timestamp())

query_response = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True,
    filter={
        "date": {"$gte": published_date_start},
    },
)

In [18]:
query_response

{'matches': [{'id': '66825e4fbd1ff5839d2dcc20',
              'metadata': {'body': 'It covers many disruptive technology and '
                                   'trends including Space, Robotics, '
                                   'Artificial Intelligence, Medicine ... A '
                                   'frequent speaker at corporations, he has '
                                   'been a TEDx speaker, a Singularity '
                                   'University speaker and guest at numerous '
                                   'interviews for radio ...',
                           'date': 1719683100.0,
                           'found_at': 1719812367.763,
                           'title': 'Supreme Court Deregulates Tesla FSD and '
                                    'Cars',
                           'url': 'https://www.nextbigfuture.com/2024/06/supreme-court-deregulates-tesla-fsd-and-cars.html'},
              'score': 0.72503275,
              'sparse_values': {'indices'

# Langchain QA


In [19]:
%pip install -qU langchain-pinecone langchain-openai langchain-anthropic langchain-voyageai langchain-text-splitters langchain

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blog-db 0.0.2 requires update<0.0.2,>=0.0.1, which is not installed.

[notice] A new release of pip is available: 24.0 -> 24.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from langchain_voyageai import VoyageAIEmbeddings

embeddings = VoyageAIEmbeddings(  # type:ignore
    model=EMBEDDING_MODEL,
)

batch size None


In [21]:
from langchain_pinecone import PineconeVectorStore


docsearch = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    text_key="title",
)

docsearch.similarity_search(QUERY, k=3)

[Document(page_content='Supreme Court Deregulates Tesla FSD and Cars', metadata={'body': 'It covers many disruptive technology and trends including Space, Robotics, Artificial Intelligence, Medicine ... A frequent speaker at corporations, he has been a TEDx speaker, a Singularity University speaker and guest at numerous interviews for radio ...', 'date': 1719683100.0, 'found_at': 1719812367.763, 'url': 'https://www.nextbigfuture.com/2024/06/supreme-court-deregulates-tesla-fsd-and-cars.html'}),
 Document(page_content="Elon Musk Suggests It's the End of the Road for the Hardware 3 Autopilot Computer", metadata={'body': 'Elon Musk said that further FSD development would require an upgraded Autopilot computer, meaning that Hardware 3 might have reached its limits', 'date': 1719834240.0, 'found_at': 1719898707.024, 'url': 'https://www.autoevolution.com/news/elon-musk-suggests-it-s-the-end-of-the-road-for-the-hardware-3-autopilot-computer-236318.html'}),
 Document(page_content='Tesla makes p

In [76]:
from langchain_core.prompts import format_document
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import PromptTemplate
from langchain_core.documents import Document
from datetime import date

from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from operator import itemgetter

gpt_3_5 = ChatOpenAI(
    model_name="gpt-3.5-turbo",  # type:ignore
    temperature=0.0,
)
haiku = ChatAnthropic(model_name="claude-3-haiku-20240307", temperature=0.0)  # type: ignore
sonnet3_5 = ChatAnthropic(model_name="claude-3-5-sonnet-20240620", temperature=0.0)  # type: ignore

llm = sonnet3_5


prompt = PromptTemplate.from_template(
    "You are an assistant for question-answering about the latest AI news. "
    "Use the following pieces of retrieved news articles to answer the question. "
    "You are talking to an experienced auidence in AI. "
    "If you don't know the answer, just say that you don't know. "
    "Format your answer in markdown and add inline hyperlinks."
    "Do not start your answer with 'Based on the provided context', or similar phrases. "
    f"Today is {date.today()} and below are the latest news on AI. \n"
    "Question : {question}\n"
    "Context : \n{context}\n\n"
    "Answer:",
)


def format_docs(docs: list[Document], separator: str = "\n\n") -> str:
    prompt = PromptTemplate.from_template(
        "{page_content} - (Published on {date})\n{url}\n{body}"
    )

    return separator.join(format_document(doc, prompt) for doc in docs)


def deduplicate_docs(docs: list[Document]) -> list[Document]:
    """Deduplicate documents based on their page_content"""

    # Create a dictionary to store unique documents
    unique_docs = {}

    for doc in docs:
        # Use the page_content as the key
        if doc.page_content not in unique_docs:
            unique_docs[doc.page_content] = doc

    # Return the list of unique documents
    return list(unique_docs.values())


retriever = docsearch.as_retriever(search_kwargs={"k": 30})

rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": itemgetter("question")
        | retriever
        | RunnableLambda(deduplicate_docs)
        | RunnableLambda(format_docs),
    }
    | prompt
    | llm
    | StrOutputParser()
)


for chunk in rag_chain.stream(
    {"question": "Quelle est la politique du gouvernement français dans l'IA ?"}
):
    print(chunk, flush=True, end="")

La politique du gouvernement français en matière d'IA s'articule autour de plusieurs axes :

1. Réglementation européenne : La France participe activement à la mise en œuvre de la réglementation européenne sur l'IA, notamment l'[IA Act](https://www.lesechos.fr/thema/articles/ia-leurope-doit-rester-dans-la-course-2105446). 

2. Développement éthique : Le gouvernement soutient le [développement d'une IA éthique et collaborative](https://www.leconomiste.com/article/1122339-intelligence-artificielle-il-n-y-pas-de-strategie-nationale), en ligne avec les valeurs démocratiques.

3. Défense et sécurité : La France investit dans l'IA pour renforcer ses capacités de défense, notamment avec le [développement du supercalculateur le plus rapide d'Europe pour la défense](https://www.msn.com/en-us/news/technology/france-preps-europes-fastest-classified-supercomputer-for-defense-ai/ar-BB1orxT5).

4. Innovation et startups : Le gouvernement soutient l'écosystème des [startups françaises en IA](https://

# TODO :

- [ ] Add rerank https://github.com/pinecone-io/examples/blob/master/learn/generation/better-rag/00-rerankers.ipynb
